In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv('/Users/ankushkumar/AI-Ml/Machine Learning/class-01/gym_members.csv')
df.head()

,member_id,age,plan_type,workouts_per_week,session_minutes,distance_km,months_active,trainer,referral_source,renewed
0,GYM3230,43.0,Pro,3,56.0,9.3,35.0,No,Walk-in,No
1,GYM3391,16.0,Basic,3,NaN,10.6,17.0,No,Friend,No
2,GYM3137,16.0,Basic,4,61.0,5.2,2.0,No,Instagram,No
3,GYM3339,28.0,Basic,5,NaN,2.5,31.0,Yes,Walk-in,Yes
4,GYM3291,32.0,Pro,6,NaN,3.3,15.0,No,Instagram,Yes


In [25]:
# Step 1 - Load and look
# Expected shape: (500, 10)

print(df.shape)
print(df.head())
print(df.info())

(500, 10)
  member_id   age plan_type  workouts_per_week  session_minutes  distance_km  \
0   GYM3230  43.0       Pro                  3             56.0          9.3   
1   GYM3391  16.0     Basic                  3              NaN         10.6   
2   GYM3137  16.0     Basic                  4             61.0          5.2   
3   GYM3339  28.0     Basic                  5              NaN          2.5   
4   GYM3291  32.0       Pro                  6              NaN          3.3   

   months_active trainer referral_source renewed  
0           35.0      No         Walk-in      No  
1           17.0      No          Friend      No  
2            2.0      No       Instagram      No  
3           31.0     Yes         Walk-in     Yes  
4           15.0      No       Instagram     Yes  
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_i

In [23]:
df.describe().columns

Index(['age', 'workouts_per_week', 'session_minutes', 'distance_km',
       'months_active'],
      dtype='str')

In [26]:
# Step 2 - Separate numeric and categorical columns
# Expected: 5 numeric, 3 categorical. member_id and renewed go in neither list.

X = df.drop(columns=['member_id', 'renewed'])
y = df['renewed']
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['age', 'workouts_per_week', 'session_minutes', 'distance_km', 'months_active']
Categorical: ['plan_type', 'trainer', 'referral_source']


/var/folders/26/8dmzs5m57bl3f9r887j35wk80000gn/T/ipykernel_98262/3201459108.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


In [21]:

# Step 3 - Split into X and y
# Expected: X (500, 8), y (500,)
x = df.drop(columns=['member_id', 'renewed'])
y = df['renewed']

print(x.shape)
print(y.shape)


(500, 8)
(500,)


In [29]:
# # Step 4 - Handle missing values
# Fill every numeric column with its own mean.
# Expected before: session_minutes 58, distance_km 21, months_active 9. After: all zero.
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].mean())

print(X.isnull().sum())




age                  0
plan_type            0
workouts_per_week    0
session_minutes      0
distance_km          0
months_active        0
trainer              0
referral_source      0
dtype: int64


In [ ]:

# Step 5 - Encode and scale together
# Use one ColumnTransformer.

# Expected shape: (500, 14)


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


cat = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)       
print(cat.fit_transform(X).shape)


(500, 14)


In [32]:
# Step 6 - Train and test split
# Expected: train (400, 14), test (100, 14)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (400, 8)
Test shape: (100, 8)


In [35]:
# Step 7 - Train the model
# KNeighborsClassifier with n_neighbors=5, then predict on the test set.

from sklearn.neighbors import KNeighborsClassifier
cat = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(cat.fit_transform(X_train), y_train)
y_pred = knn.predict(cat.fit_transform(X_test))


In [36]:
 # Step 8 - Score
df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})
print(df.head())

    y_test y_pred
361     No     No
73      No     No
374    Yes    Yes
155    Yes    Yes
104    Yes    Yes
